## Steps to Create a Resource Group
* Go to the Azure portal and search for Resource Group.
* Click on Create.
* In the Resource Group name field, enter `RGmodernDB`.
* Click Review + create.
* Click Create again to deploy it.

## Steps to Create a Storage Account (Data Lake)
* Click on Create a resource and search for Storage account.
* Click on Create.
* Ensure the Resource Group is set to `RGmodernDB`.
* In the Storage account name field, enter `storagemodernDB`.
* Select Standard for performance.
* For Redundancy, select LRS (Locally-redundant storage) to keep the cost low.
* Click Next to proceed to the Advanced tab.
* Check the box for Enable hierarchical namespace (this converts the storage into a Data Lake).
* Click Review + create.
* Click Create to deploy it.
* Create two containers `metastoreroot` and `mycontainer`.

## Steps to Create Azure Databricks
* Click on Create a resource and search for Azure Databricks.
* Click on Create.
* Ensure the Resource Group is set to `RGmodernDB`.
* In the Workspace name field, enter `ADBmodern`.
* For Pricing Tier, select Trial (Premium - 14-Days Free DBUs).
* Scroll down to the Managed Resource Group field and name it `RGmodernDBmanaged` so it is easy to identify.
* Click Review + create.
* Click Create to deploy the workspace.

## Creating an Access Connector for Azure Databricks
**Access Connector:** An Azure resource that provides a managed identity. It acts as an "ID card" allowing the Unity Metastore to securely read and write data in your Data Lake.

**Steps to Create:**
* Search for **Access Connector for Azure Databricks** in the Azure Portal.
* Click **Create**.
* Select your Resource Group and name the connector (e.g., `accessModernDB`).
* Click **Review + create**, then **Create**.

**Granting Permissions to the Connector:**
* Go back to your Storage Account.
* Click on **Access Control (IAM)** in the left menu.
* Click **Add** -> **Add role assignment**.
* Search for and select the **Storage Blob Data Contributor** role. Click Next.
* Select assign access to **Managed identity**.
    - **Select your resource type by choosing "Access connector for Azure Databricks" from the dropdown.**
    - Select your newly created Access Connector (e.g., `access modern DB`) and click **Select**.
    - Click **Review + assign** to finish.
* Select your newly created Access Connector and assign the role.

* **Storage Blob Data Contributor:** Required for reading and writing data.
* **Storage Queue Data Contributor:** Required for subscribing to file event notifications.
* **Storage Account Contributor:** Required if you want Azure Databricks to automatically configure file events. If you do not grant this role, you must manually create the Azure storage queue and configure the event subscription yourself. Manual configuration is a self-service process, and Databricks does not provide support for setting up these resources.
* **EventGrid EventSubscription Contributor:** Required if you want Azure Databricks to automatically configure file events. If you do not grant this role, you must manually configure the event subscription yourself. Manual configuration is a self-service process, and Databricks does not provide support for setting up these resources.

## Metastore Concepts
**Hive Metastore:** The legacy, default metadata repository. Historically, every Databricks workspace had its own independent Hive Metastore.
**Unity Metastore:** A centralized data governance solution that stores metadata for multiple Databricks workspaces under one hood. You can only create one Unity Metastore per cloud region.

## Creating the Unity Metastore
**Steps to Create:**
* Navigate to the Databricks Account Console by going to `https://accounts.azuredatabricks.net/login` (you may need to sign in with your Microsoft Entra ID).
    - Go to the Azure portal homepage.
    - Click on **Microsoft Entra ID**.
    - Under the **Manage** tab on the left menu, click on **Users**.
    - Select your user account and copy the email ID listed there to use for sign-in.
* Go to the **Catalog** tab.
* Click **Create Metastore**.
* **Name:** Enter a name for the metastore (e.g., `modernmetastore`).
* **Region:** Select the same region where your workspace is located.
* **ADLS Gen 2 path:** Enter the path to the dedicated container directory using the syntax: (`abfss://metastoreroot@storagemoderndbaps.dfs.core.windows.net/meta-directory/`).
* **Access Connector ID:** Paste the Resource ID of the Access Connector you created (found in the properties of the Access Connector in Azure).
* **User assigned managed identity ID:** If prompted, paste the Resource ID of your user-assigned managed identity. To set this up:
  - In the Azure portal, search for **Managed Identities** and click **Create**.
  - Go back to your Access Connector for Azure Databricks (e.g., `accessModernDB`).
  - In the left menu under **Settings**, click on **Identity**.
  - Select the **User assigned** tab and click **Add** to attach the managed identity you just created.
  - Copy the Resource ID of this new managed identity to use for this step.
* Click **Create**.

## Assigning the Metastore to a Workspace
* Immediately after creation, a prompt will ask you to assign the metastore to a workspace.
* Select your Databricks workspace from the list.
* Click **Assign**, then click **Enable**. This officially connects your workspace to Unity Catalog.

## Compute (Cluster) Concepts
**Compute (Cluster):** A configuration of virtual machines (either a single node or multiple interconnected nodes) used to execute data processing workloads. When you create a cluster in the Databricks UI, the actual underlying virtual machines are automatically provisioned inside your Managed Resource Group in Azure.

## Steps to Create Compute
* Navigate to your Databricks workspace and click on the **Compute** tab in the left sidebar.
* Click the **Create compute** button.
* **Databricks runtime version:** Ensure the selected runtime indicates it is enabled with Unity Catalog (this is selected by default once your workspace is linked to the metastore).
* **Cluster mode:** Select **Single node** instead of multi-node to keep costs low for learning and development.
* **Node type:** Select a standard, basic virtual machine type, such as **Standard_DS3_v2**.
* **Auto termination:** Ensure this option is enabled and set the cluster to terminate after **20 minutes** of inactivity to save resources.
* Click **Create compute**. 

*Note: Once the compute is successfully created, you can return to the Azure Portal and open your Managed Resource Group to see the actual virtual machine that Databricks provisioned.*

## Granting Metastore Admin Privileges
Before creating a catalog, the user must be assigned as a Metastore Admin; otherwise, the option to create a catalog will not be visible in the workspace.

*Steps:*
* Navigate to the Databricks Account Console (`accounts.azuredatabricks.net`).
* Go to the **Catalog** tab and click on your Unity Metastore.
* Click **Edit** next to the Metastore Admin section.
* Select your Microsoft Entra ID account from the list.
* Click **Save**.

## Creating a Folder and Notebook
**Workspace:** The area in Databricks where you organize your folders and notebooks.

*Steps to Create:*
* In the Databricks UI, click on the **Workspace** tab in the left sidebar.
* Click on **Create** -> **Folder**.
* Name the folder (e.g., `modern DB`) and click **Create**.
* Open the newly created folder, click **Create** -> **Notebook**.
* Enter a name for the notebook (e.g., `tutorial one`).
* **Attach to Cluster:** In the top left corner of the notebook, click the cluster dropdown menu and attach the notebook to your previously created compute cluster to enable code execution.

## Magic Commands
Magic commands allow you to use different programming languages or functionalities within specific cells of a single notebook. The default language is usually Python, but you can override it per cell.

*Common Magic Commands:*
* `%python`: Executes the cell as Python code.
* `%sql`: Executes the cell as SQL code.
* `%scala`: Executes the cell as Scala code.
* `%r`: Executes the cell as R code.
* `%md` (Markdown): Executes the cell as Markdown for writing text, creating headings, or adding comments.
* `%fs`: Used to access the Databricks file system (e.g., testing connections or listing files). Alternatively, `dbutils.fs` can be used in Python.

## Storage Credential
**Storage Credential:** A Unity Catalog authentication object that acts as an "ID card." It stores the Access Connector ID (Managed Identity) and grants Databricks the necessary permissions to access the connected cloud storage. A storage credential is a prerequisite for creating an external location.

*Steps to Create:*
* In the Databricks workspace, go to the **Catalog** tab.
* Click the **+** (Add) button or navigate to **External Data**.
* Select **Credentials** and click **Create credential**.
* **Name:** Enter a unique name for the credential.
* **Access Connector ID:** Paste the exact Resource ID of your previously created Azure Access Connector.
* Click **Create**.

*Steps to Grant Permissions (UI):*
- After creating the credential, click on the newly created **Storage Credential** to open its details page.
- Go to the **Permissions** tab.
- Click the **Grant** button.
- Under **Principals**, search for and select the user (using their email ID) or group you want to grant access to.
- Under **Privilege presets** or the privileges list, check **ALL PRIVILEGES** (or select specific privileges like *CREATE EXTERNAL LOCATION*, *READ FILES*, and *WRITE FILES*).
- Click **Grant** to save the changes.

*Granting Permissions (via SQL):*
- Alternatively, you can grant all privileges on the storage credential using a SQL notebook or the SQL Editor:
```sql
GRANT ALL PRIVILEGES ON STORAGE CREDENTIAL `your_credential_name` TO `user.email@domain.com`;
```

## External Location
**External Location:** A Unity Catalog object that securely associates a specific cloud storage path (such as an Azure Data Lake container) with a Storage Credential. This authorizes users to read, write, and manage external tables or files in that specific cloud storage path.

*Pro Tip:* It is highly recommended to create the external location at the **container level** rather than the folder level. This allows you to use the same external location for any new folders you create inside that container later.

*Steps to Create:*
* In the Databricks workspace, go to the **Catalog** tab.
* Click the **+** (Add) button or navigate to **External Data**.
* Select **External Locations** and click **Create external location**.
* **Name:** Enter a name for the external location `myextloc`.
* **URL:** Provide the ADLS Gen2 storage path using the format: `abfss://container-name@storage-account-name.dfs.core.windows.net/` (Make sure to include the trailing slash).- abfss://mycontainer@storagemoderndbaps.dfs.core.windows.net/
* **Storage credential:** Select the Storage Credential you created in the previous step from the dropdown list.
* Click **Create**.

## Creating a Managed Catalog
A managed catalog does not have a specific storage location defined. By default, any managed schemas or tables created inside it will be stored in the root storage location dedicated to the Unity Metastore.

*Steps (via Notebook SQL):*
* Open a Databricks Notebook.
* Ensure the cell language is set to SQL (using `%sql` if it is a Python notebook).
* Run the following SQL command:
  `CREATE CATALOG catalog_name;`

## Creating an External Catalog
An external catalog is assigned its own specific storage location. Any managed schemas or tables created inside this catalog will default to this specific location instead of the Unity Metastore root storage.

*Steps (via Notebook SQL):*
* Run the following SQL command, explicitly specifying your custom storage path:
  `CREATE CATALOG catalog_name MANAGED LOCATION 'abfss://container-name@storage-account-name.dfs.core.windows.net/folder-name/';` 

## Databricks Real-Time Scenarios

### Scenario 1: Managed Catalog, Managed Schema, Managed Table
* **Concept:** If no location is explicitly provided at any level, Unity Catalog defaults to the root storage location dedicated to the Unity Metastore. 
* **Hierarchy Flow:** The managed table takes the location of the managed schema -> which takes the location of the managed catalog -> which takes the Metastore root location.
* **SQL Example:**
```sql
CREATE CATALOG man_cata;
CREATE SCHEMA man_cata.man_schema;

CREATE TABLE man_cata.man_schema.man_table (
  ID INT,
  name STRING
) USING DELTA;
```

### Scenario 2: External Catalog, Managed Schema, Managed Table

* **Concept:** An external catalog is assigned a custom `MANAGED LOCATION`. Any managed schemas and tables created inside it will bypass the Metastore root and use this catalog's custom location instead.
* **SQL Example:**

    ```sql
    CREATE CATALOG ext_cata 
    MANAGED LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/external_catalog/';
    
    CREATE SCHEMA ext_cata.man_schema;
    
    CREATE TABLE ext_cata.man_schema.man_table2 (
      ID INT,
      name STRING
    ) USING DELTA;
    ```
### Scenario 3: External Catalog, External Schema, Managed Table

* **Concept:** You can define a specific `MANAGED LOCATION` at the schema level. When creating a managed table, the storage location priority is evaluated from the bottom up (Schema -> Catalog). The table will be stored in the schema's location.
* **SQL Example:**

    ```sql
    CREATE SCHEMA ext_cata.ext_schema 
    MANAGED LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/external_schema/';
    
    CREATE TABLE ext_cata.ext_schema.man_table3 (
      ID INT,
      name STRING
    ) USING DELTA;
    ```

### Scenario 4: External Catalog, External Schema, External Table

* **Concept:** An external table uses the `LOCATION` keyword explicitly during table creation. It completely ignores the parent locations (even if the catalog or schema is managed or external) and stores data exactly in the specific path provided.
* **SQL Example:**

    ```sql
    CREATE TABLE ext_cata.ext_schema.ext_table (
      ID INT,
      name STRING
    ) USING DELTA
    LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/ext_table/';
    ```
### DROP and UNDROP in Databricks

* **Dropping a Managed Table:** When you drop a managed table in Unity Catalog, the metadata is deleted immediately, but the underlying data is retained for **7 days** before being permanently purged.
* **UNDROP Command:** Because the data persists for 7 days, you can recover a dropped managed table (and its data) using the `UNDROP` command if you made a mistake.
* **SQL Example:**

    ```sql
    DROP TABLE man_cata.man_schema.man_table;
    
    UNDROP TABLE man_cata.man_schema.man_table;
    ```
### Insert Values into a Table

* **Concept:** You can populate your Delta tables with data using standard SQL `INSERT INTO` statements.
* **SQL Example:**

    ```sql
   INSERT INTO ext_cata.man_schema.man_table (id, name) 
    VALUES 
      (1, 'AA'),
      (2, 'BB'),
      (3, 'CC');
    ```
### Databricks Views

* **Direct File Querying:** You can query data directly from a storage path without creating a table by specifying the file format (e.g., `delta`) and the path.
* **SQL Example:**

    ```sql
    SELECT * FROM delta.`abfss://mycontainer@storage_account_name.dfs.core.windows.net/ext_table/`;
    ```

* **Permanent Views:** Saved in the metastore and require a Catalog and Schema namespace. They persist permanently, even if the cluster is detached or restarted.
* **SQL Example:**

    ```sql
    CREATE VIEW man_cata.man_schema.view1 AS 
    SELECT * FROM delta.`abfss://mycontainer@storage_account_name.dfs.core.windows.net/ext_table/man_table` 
    WHERE ID = 1;
    ```

* **Temporary Views:** Tied exclusively to the active Spark session. They do not require a catalog or schema namespace and will be automatically deleted if the cluster is detached or restarted.
* **SQL Example:**

    ```sql
    CREATE OR REPLACE TEMP VIEW temp_view1 AS 
    SELECT * FROM delta.`abfss://mycontainer@storage_account_name.dfs.core.windows.net/ext_table/`;

### Read Data from ADLS using Databricks

* **Concept:** You can directly query raw files (like Delta, Parquet, CSV, or JSON) stored in ADLS without creating a table or view first. You just need to specify the file format and the direct cloud storage path wrapped in backticks.
* **SQL Example:**

    ```sql
    SELECT * FROM delta.`abfss://mycontainer@storage_account_name.dfs.core.windows.net/external_table/man_table_3/`;
    ```
### Databricks Volumes in Unity Catalog

* **Concept:** Volumes are Unity Catalog objects used to store, access, and govern non-tabular, semi-structured, or unstructured data (like CSV, Parquet, JSON, or images). Just like tables, you can create both managed and external volumes.
* **SQL Example:**

    ```sql
    CREATE EXTERNAL VOLUME man_cata.man_schema.external_volume
    LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/volumes/';
    ```
### Managed Volumes and External Volumes in Databricks

* **Concept:** Volumes in Unity Catalog govern non-tabular data (like CSV, images, or JSON). A **Managed Volume** stores files in the default schema or catalog location, while an **External Volume** stores files in a custom, explicitly defined cloud storage path.
* **Create a Source Directory and Upload Data:** In your Azure Data Lake container, manually create a directory named `Source` and upload a sample file (e.g., `sales.csv`) into it.
* **Step 1: Creating a Directory for the Volume:** Before creating an external volume, you can use Databricks utilities (Python) to create a dedicated directory in your Data Lake.

    ```python
    dbutils.fs.mkdirs("abfss://mycontainer@storage_account_name.dfs.core.windows.net/volumes/")
    ```

* **Step 2: Creating the Volumes (SQL):** Once the directory is ready, you can create the volume.

    ```sql
    -- Creating a Managed Volume
    CREATE VOLUME man_cata.man_schema.managed_volume;
    
    -- Creating an External Volume
    CREATE EXTERNAL VOLUME man_cata.man_schema.external_volume
    LOCATION 'abfss://mycontainer@storage_account_name.dfs.core.windows.net/volumes/';
    ```

* **Step 3: Copying a File into the Volume:** You can move or copy files directly into your new volume directory using `dbutils`.

    ```python
    dbutils.fs.cp(
        "abfss://mycontainer@storage_account_name.dfs.core.windows.net/Source/sales.csv", 
        "abfss://mycontainer@storage_account_name.dfs.core.windows.net/volumes/sales.csv"
    )
    ```
* **Step 4: Querying the Volume:** Once the file is in the volume, you can query the data inside it directly using its path and format.

    ```sql
    SELECT * FROM csv.`abfss://mycontainer@storage_account_name.dfs.core.windows.net/volumes/sales.csv`;
    ```